
# V3-SEA-8 Dataset Workshop - Seasonal `rx1day` Percentage Change Maps

This notebook demonstrates how to use V3-SEA-8 `rx1day` data to evaluate seasonal changes in extreme daily rainfall over Malaysia.

`rx1day` represents the maximum daily rainfall amount recorded within each month. For this workshop, we convert the monthly data into seasonal values using this logic:

1. Select one season: DJF, MAM, JJA, or SON.
2. For each year and grid cell, take the maximum `rx1day` value among the three months in that season.
3. Average those annual seasonal maxima over the historical baseline period, 1995-2014.
4. Repeat for the end-century future period, 2080-2099, under SSP5-8.5.
5. For each model, calculate percentage change first:

```text
((future seasonal climatology - historical seasonal climatology) / historical seasonal climatology) * 100
```

6. Average the model-level percentage-change maps across the selected models.

The final plot contains four seasonal maps showing ensemble-mean percentage change for DJF, MAM, JJA, and SON.



## 0. Local IDE Setup

This notebook is designed for a local Jupyter environment, such as VS Code, JupyterLab, or classic Jupyter Notebook. Run the setup instructions in `README.md` first, then select the correct kernel and run the cells in order.


In [ ]:

import sys
from pathlib import Path

WORKSHOP_PATH = Path.cwd().resolve()
if str(WORKSHOP_PATH) not in sys.path:
    sys.path.insert(0, str(WORKSHOP_PATH))

required_packages = {
    "xarray": "xarray",
    "netCDF4": "netCDF4",
    "geopandas": "geopandas",
    "shapely": "shapely",
    "dask": "dask",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
}

missing_packages = []
for package_name, import_name in required_packages.items():
    try:
        __import__(import_name)
    except ImportError:
        missing_packages.append(package_name)

if missing_packages:
    raise ModuleNotFoundError(
        "Missing required package(s): "
        + ", ".join(missing_packages)
        + ". Install them with `python -m pip install -r requirements.txt` or `conda env create -f environment.yml`."
    )

print("Local workshop setup looks ready.")
print(f"  Working directory: {WORKSHOP_PATH}")
print(f"  Python executable: {sys.executable}")
print("  Required packages can be imported.")



## 1. Import Libraries And Configuration Settings

Change `REGION_KEY` and `MODELS_TO_RUN` to control the runtime and the plotted area:

- Use `REGION_KEY = "malaysia"` for Peninsular Malaysia + Sabah + Sarawak together.
- Use `REGION_KEY = "penisular"`, `"sabah"`, or `"sarawak"` to focus on one region.
- Use one model for a fast first run.
- Use two or more models to show an ensemble-mean result.
- Use `MODELS_TO_RUN = "all"` to process every available model.


In [ ]:

from __future__ import annotations

import math
import os
import re
from functools import lru_cache
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
import numpy as np
import pandas as pd
from shapely import contains_xy
import xarray as xr

# Load validated local paths from config.py.
try:
    from config import CCRS_DATA_PATH, SHAPEFILES_PATH
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "config.py was not found. Start Jupyter from the repository folder that contains this notebook and config.py."
    ) from exc

WORKSHOP_DIR = Path.cwd()
CCRS_ROOT = CCRS_DATA_PATH
SHAPES_ROOT = SHAPEFILES_PATH
OUTPUT_DIR = WORKSHOP_DIR / "outputs" / "rx1day"

REGION_KEY = "malaysia"  # choose: "malaysia", "penisular", "sabah", or "sarawak"
MODELS_TO_RUN = ["ACCESS-CM2"]  # use one model, multiple models, or "all"
FUTURE_SCENARIO = "ssp585"
CHUNKS_TIME = 12
USE_CACHE = True
CACHE_VERSION = "seasonal_shape_mask_v1"

HISTORICAL_PERIOD = (1995, 2014)
FUTURE_PERIOD = (2080, 2099)
BOUNDARY_TOLERANCE_DEGREES = 0.015

SEASONS = {
    "DJF": (12, 1, 2),
    "MAM": (3, 4, 5),
    "JJA": (6, 7, 8),
    "SON": (9, 10, 11),
}

REGION_FOLDERS = {
    "penisular": ("PENINSULAR_MALAYSIA",),
    "sabah": ("SABAH",),
    "sarawak": ("SARAWAK",),
    "malaysia": ("PENINSULAR_MALAYSIA", "SABAH", "SARAWAK"),
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("RX1day workshop configuration loaded.")
print(f"  Region: {REGION_KEY}")
print(f"  Models: {MODELS_TO_RUN}")
print(f"  CCRS root: {CCRS_ROOT}")
print(f"  Shapefile root: {SHAPES_ROOT}")
print(f"  Output directory: {OUTPUT_DIR}")



## 2. Helper Functions For Data Discovery

The functions below find model folders, select the models requested in `MODELS_TO_RUN`, and collect only the RX1day files needed for the target period.


In [ ]:

def all_model_dirs(ccrs_root: Path) -> list[Path]:
    return sorted(path for path in ccrs_root.iterdir() if path.is_dir())


def selected_model_dirs(ccrs_root: Path, models_to_run: str | list[str]) -> list[Path]:
    available = {path.name: path for path in all_model_dirs(ccrs_root)}
    if models_to_run == "all":
        return [available[name] for name in sorted(available)]

    missing = [name for name in models_to_run if name not in available]
    if missing:
        raise FileNotFoundError(f"Selected model folder(s) not found: {missing}")
    return [available[name] for name in models_to_run]


def year_from_filename(path: Path) -> int:
    match = re.search(r"_(\d{4})\d{2}-\d{4}\d{2}\.nc$", path.name)
    if not match:
        raise ValueError(f"Cannot parse year from {path}")
    return int(match.group(1))


def scenario_files(
    model_dir: Path,
    scenario: str,
    variable: str,
    start_year: int,
    end_year: int,
) -> list[Path]:
    files = sorted((model_dir / scenario).glob(f"**/{variable}/*.nc"))
    return [path for path in files if start_year <= year_from_filename(path) <= end_year]


def coord_slice(coord: xr.DataArray, lower: float, upper: float) -> slice:
    if float(coord[0]) > float(coord[-1]):
        return slice(upper, lower)
    return slice(lower, upper)


def safe_name(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", value)


models = selected_model_dirs(CCRS_ROOT, MODELS_TO_RUN)
print("Selected models:")
for model in models:
    print(f"  - {model.name}")



## 3. Inspect The RX1day File Structure

Before the full calculation, inspect how many files are available for each selected model and open one sample file to confirm dimensions, coordinates, and units.


In [ ]:
for model in models:
    historical_files = scenario_files(model, "historical", "rx1day", *HISTORICAL_PERIOD)
    future_files = scenario_files(model, FUTURE_SCENARIO, "rx1day", *FUTURE_PERIOD)
    print(
        f"{model.name}: "
        f"historical {HISTORICAL_PERIOD[0]}-{HISTORICAL_PERIOD[1]} = {len(historical_files)} files, "
        f"{FUTURE_SCENARIO} {FUTURE_PERIOD[0]}-{FUTURE_PERIOD[1]} = {len(future_files)} files"
    )

sample_files = scenario_files(models[0], "historical", "rx1day", *HISTORICAL_PERIOD)
if not sample_files:
    raise FileNotFoundError(f"No historical RX1day files found for {models[0].name}")

sample_file = sample_files[0]
print(f"\nSample file: {sample_file}")
with xr.open_dataset(sample_file, engine="netcdf4") as sample_ds:
    display(sample_ds)



## 4. Load And Merge The Region Shapefiles

The geometry is cached in memory so repeated mask operations do not keep re-reading and re-merging the same shapefiles.


In [ ]:

def shapefile_paths_for_region(region_key: str, shapes_root: Path) -> list[Path]:
    if region_key not in REGION_FOLDERS:
        raise KeyError(f"Unknown region {region_key!r}. Choose from {sorted(REGION_FOLDERS)}")

    paths: list[Path] = []
    for folder_name in REGION_FOLDERS[region_key]:
        folder = shapes_root / folder_name
        if not folder.exists():
            raise FileNotFoundError(f"Required shapefile folder not found: {folder}")
        paths.extend(sorted(folder.glob("*.shp")))

    if not paths:
        raise FileNotFoundError(f"No .shp files found for region {region_key!r} under {shapes_root}")
    return paths


def union_geometries(gdf: gpd.GeoDataFrame):
    if hasattr(gdf.geometry, "union_all"):
        return gdf.geometry.union_all()
    return gdf.geometry.unary_union


@lru_cache(maxsize=8)
def cached_region_geometry(region_key: str, shapes_root_text: str, tolerance: float):
    shapes_root = Path(shapes_root_text)
    geometries = []

    for shp_path in shapefile_paths_for_region(region_key, shapes_root):
        gdf = gpd.read_file(shp_path)
        if gdf.empty:
            continue
        if gdf.crs is None:
            gdf = gdf.set_crs("EPSG:4326")
        else:
            gdf = gdf.to_crs("EPSG:4326")
        geometries.append(union_geometries(gdf))

    if not geometries:
        raise ValueError(f"No usable geometries found for region {region_key!r}")

    geometry = union_geometries(gpd.GeoDataFrame(geometry=geometries, crs="EPSG:4326"))
    geometry = geometry.buffer(0)
    if tolerance > 0:
        geometry = geometry.buffer(tolerance).buffer(-tolerance)
    return geometry


def region_geometry(region_key: str = REGION_KEY):
    return cached_region_geometry(
        region_key,
        str(SHAPES_ROOT.resolve()),
        BOUNDARY_TOLERANCE_DEGREES,
    )


geometry = region_geometry(REGION_KEY)
print(f"Loaded geometry for {REGION_KEY!r}")
print(f"  Bounds: {tuple(round(value, 3) for value in geometry.bounds)}")
print(f"  Area in degree units: {geometry.area:.3f}")



## 5. Build The Shapefile Mask

The climate grid is first cropped to the selected region bounds, then each grid-cell centre is tested against the shapefile geometry. This ensures the spatial mean and maps use the shapefile-defined region rather than a rectangular latitude/longitude box.


In [ ]:

def mask_region(da: xr.DataArray, region_key: str = REGION_KEY) -> xr.DataArray:
    geometry = region_geometry(region_key)
    mask_geometry = geometry.buffer(1e-12)
    lon_min, lat_min, lon_max, lat_max = geometry.bounds

    lat_step = float(np.abs(np.diff(da.lat.values)).min()) if da.sizes["lat"] > 1 else 0.0
    lon_step = float(np.abs(np.diff(da.lon.values)).min()) if da.sizes["lon"] > 1 else 0.0

    cropped = da.sel(
        lat=coord_slice(da.lat, lat_min - lat_step, lat_max + lat_step),
        lon=coord_slice(da.lon, lon_min - lon_step, lon_max + lon_step),
    )

    lon2d, lat2d = np.meshgrid(cropped.lon.values, cropped.lat.values)
    inside = contains_xy(mask_geometry, lon2d, lat2d)
    if int(inside.sum()) == 0:
        raise ValueError("Region shapefile mask did not select any grid cells.")

    mask = xr.DataArray(
        inside,
        coords={"lat": cropped.lat, "lon": cropped.lon},
        dims=("lat", "lon"),
        name=f"{region_key}_mask",
    )
    return cropped.where(mask)


with xr.open_dataset(sample_file, engine="netcdf4") as sample_ds:
    masked_sample = mask_region(sample_ds["rx1day"].isel(time=0), REGION_KEY)

masked_sample.plot(figsize=(7, 3.5))
plt.title(f"Example masked RX1day grid: {REGION_KEY}")
plt.show()



## 6. Compute Seasonal RX1day Climatology

For each season, the notebook selects the relevant months and takes the maximum RX1day value per year. It then averages those annual seasonal maxima across the target period.

Dask is used through `xarray.open_mfdataset(..., chunks={"time": CHUNKS_TIME})`, so the multi-file NetCDF data are opened lazily and only computed when needed.


In [ ]:

def convert_rx1day_units(da: xr.DataArray) -> xr.DataArray:
    units = str(da.attrs.get("units", "")).lower().strip()
    if units in {"kg m-2 s-1", "kg m**-2 s**-1", "kg/m2/s"}:
        da = da * 86400.0
        da.attrs["units"] = "mm/day"
    return da


def seasonal_period_rx1day_maps(
    files: list[Path],
    region_key: str,
    chunks_time: int = CHUNKS_TIME,
) -> xr.DataArray:
    ds = xr.open_mfdataset(
        [str(path) for path in files],
        combine="by_coords",
        chunks={"time": chunks_time},
        data_vars="minimal",
        coords="minimal",
        compat="override",
        parallel=True,
        engine="netcdf4",
    )
    try:
        rx1day = mask_region(ds["rx1day"], region_key)
        rx1day = convert_rx1day_units(rx1day)

        seasonal_maps = []
        for season, months in SEASONS.items():
            season_rx1day = rx1day.where(rx1day["time"].dt.month.isin(months), drop=True)
            annual_season_max = season_rx1day.groupby("time.year").max("time", skipna=True)
            period_mean = annual_season_max.mean("year", skipna=True)
            seasonal_maps.append(period_mean.expand_dims(season=[season]))

        result = xr.concat(seasonal_maps, dim="season")
        result = result.sel(season=list(SEASONS))
        result.name = "rx1day"
        result.attrs["units"] = "mm/day"
        return result.compute()
    finally:
        ds.close()



## 7. Cache Or Compute Model Seasonal Change

The first run can be slow because each model requires shapefile masking and NetCDF computation. The notebook writes one small NetCDF cache per model and period so later layout changes can reuse the processed maps.


In [ ]:

def safe_to_netcdf(data: xr.DataArray | xr.Dataset, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_name(f"{path.stem}.tmp-{os.getpid()}{path.suffix}")
    data.to_netcdf(temp_path)
    temp_path.replace(path)


def load_cached_dataarray(path: Path) -> xr.DataArray:
    with xr.open_dataarray(path) as da:
        return da.load()


def seasonal_cache_paths(model_name: str, region_key: str) -> tuple[Path, Path, Path]:
    cache_dir = OUTPUT_DIR / "cache"
    cache_dir.mkdir(parents=True, exist_ok=True)
    model = safe_name(model_name)
    region = safe_name(region_key)
    tolerance = str(BOUNDARY_TOLERANCE_DEGREES).replace(".", "p")
    prefix = f"{region}_{CACHE_VERSION}_tol{tolerance}_{model}"
    return (
        cache_dir / f"{prefix}_historical_{HISTORICAL_PERIOD[0]}_{HISTORICAL_PERIOD[1]}.nc",
        cache_dir / f"{prefix}_{FUTURE_SCENARIO}_{FUTURE_PERIOD[0]}_{FUTURE_PERIOD[1]}.nc",
        cache_dir / f"{prefix}_{FUTURE_SCENARIO}_percent_change.nc",
    )


def compute_or_load_model_change(model_dir: Path, region_key: str) -> tuple[xr.DataArray, xr.DataArray, xr.DataArray]:
    historical_cache, future_cache, percent_cache = seasonal_cache_paths(model_dir.name, region_key)

    if USE_CACHE and historical_cache.exists() and future_cache.exists() and percent_cache.exists():
        print(f"Loading cached seasonal maps for {model_dir.name}")
        return (
            load_cached_dataarray(historical_cache),
            load_cached_dataarray(future_cache),
            load_cached_dataarray(percent_cache),
        )

    historical_files = scenario_files(model_dir, "historical", "rx1day", *HISTORICAL_PERIOD)
    future_files = scenario_files(model_dir, FUTURE_SCENARIO, "rx1day", *FUTURE_PERIOD)
    if not historical_files or not future_files:
        raise FileNotFoundError(
            f"Missing paired historical/{FUTURE_SCENARIO} RX1day files for {model_dir.name}"
        )

    print(f"Processing seasonal RX1day for {model_dir.name}")
    historical = seasonal_period_rx1day_maps(historical_files, region_key, CHUNKS_TIME)
    future = seasonal_period_rx1day_maps(future_files, region_key, CHUNKS_TIME)
    percent_change = xr.where(historical > 0, ((future - historical) / historical) * 100.0, np.nan)
    percent_change.name = "rx1day_seasonal_percent_change"
    percent_change.attrs["units"] = "%"

    if USE_CACHE:
        safe_to_netcdf(historical, historical_cache)
        safe_to_netcdf(future, future_cache)
        safe_to_netcdf(percent_change, percent_cache)

    return historical, future, percent_change


def ensemble_seasonal_change(model_dirs_to_run: list[Path], region_key: str):
    historical_maps = []
    future_maps = []
    percent_change_maps = []
    used_models = []
    skipped_models = []

    for model_dir in model_dirs_to_run:
        historical_files = scenario_files(model_dir, "historical", "rx1day", *HISTORICAL_PERIOD)
        future_files = scenario_files(model_dir, FUTURE_SCENARIO, "rx1day", *FUTURE_PERIOD)
        if not historical_files or not future_files:
            skipped_models.append(model_dir.name)
            continue

        historical, future, percent_change = compute_or_load_model_change(model_dir, region_key)
        historical_maps.append(historical)
        future_maps.append(future)
        percent_change_maps.append(percent_change)
        used_models.append(model_dir.name)

    if not percent_change_maps:
        raise ValueError("No selected models have both historical and SSP585 RX1day files.")

    model_index = pd.Index(used_models, name="model")
    historical_ensemble = xr.concat(historical_maps, dim=model_index).mean("model", skipna=True)
    future_ensemble = xr.concat(future_maps, dim=model_index).mean("model", skipna=True)
    percent_change_ensemble = xr.concat(percent_change_maps, dim=model_index).mean("model", skipna=True)
    percent_change_ensemble = percent_change_ensemble.sel(season=list(SEASONS))
    percent_change_ensemble.name = "rx1day_seasonal_percent_change"
    percent_change_ensemble.attrs["units"] = "%"

    return historical_ensemble, future_ensemble, percent_change_ensemble, used_models, skipped_models



## 8. Run The Seasonal RX1day Calculation

This is the main computation cell. It may take time on the first run. Later runs will reuse cached per-model NetCDF outputs when `USE_CACHE = True`.


In [ ]:
seasonal_historical, seasonal_future, seasonal_change, used_models, skipped_models = ensemble_seasonal_change(
    models,
    REGION_KEY,
)

print("\nModels used:", ", ".join(used_models))
if skipped_models:
    print("Skipped models:", ", ".join(skipped_models))

final_ds = xr.Dataset(
    {
        "historical_1995_2014_seasonal_rx1day": seasonal_historical,
        "ssp585_2080_2099_seasonal_rx1day": seasonal_future,
        "ssp585_2080_2099_seasonal_percent_change": seasonal_change,
    },
    attrs={
        "description": (
            "Seasonal Malaysia RX1day outputs. Percentage change is computed per model first, "
            "then averaged across selected models."
        ),
        "region_key": REGION_KEY,
        "models_used": ", ".join(used_models),
        "historical_period": f"{HISTORICAL_PERIOD[0]}-{HISTORICAL_PERIOD[1]}",
        "future_period": f"{FUTURE_PERIOD[0]}-{FUTURE_PERIOD[1]}",
        "future_scenario": FUTURE_SCENARIO,
        "boundary_tolerance_degrees": BOUNDARY_TOLERANCE_DEGREES,
    },
)

final_nc = OUTPUT_DIR / f"rx1day_{REGION_KEY}_seasonal_final_maps.nc"
safe_to_netcdf(final_ds, final_nc)
print(f"Wrote {final_nc}")

seasonal_change



## 9. Summarize Seasonal Change Statistics

The text and CSV summaries record the minimum, mean, and maximum percentage change for each season. The most negative grid-cell location is also included for quick interpretation.


In [ ]:

def change_stats_table(da: xr.DataArray) -> pd.DataFrame:
    rows = []
    for season in SEASONS:
        season_da = da.sel(season=season)
        values = season_da.values
        finite = values[np.isfinite(values)]
        if finite.size == 0:
            rows.append({"season": season, "min_percent": np.nan, "mean_percent": np.nan, "max_percent": np.nan})
            continue

        stacked = season_da.stack(point=("lat", "lon"))
        min_index = int(np.nanargmin(stacked.values))
        min_point = stacked.isel(point=min_index)
        rows.append(
            {
                "season": season,
                "min_percent": float(finite.min()),
                "mean_percent": float(finite.mean()),
                "max_percent": float(finite.max()),
                "most_negative_lon": float(min_point.lon),
                "most_negative_lat": float(min_point.lat),
            }
        )
    return pd.DataFrame(rows)


stats = change_stats_table(seasonal_change)
stats_csv = OUTPUT_DIR / f"rx1day_{REGION_KEY}_seasonal_change_statistics.csv"
stats_txt = OUTPUT_DIR / f"rx1day_{REGION_KEY}_seasonal_change_statistics.txt"
stats.to_csv(stats_csv, index=False)

lines = [
    f"RX1day seasonal SSP585 percentage-change statistics for {REGION_KEY}",
    "Percentage change = ((future climatology - historical climatology) / historical climatology) * 100",
    "Percentage change is computed per model first, then averaged across selected models.",
    f"Models used: {', '.join(used_models)}",
    "",
]
for _, row in stats.iterrows():
    lines.extend(
        [
            f"{row['season']}:",
            f"  min_percent_change = {row['min_percent']:.2f}",
            f"  mean_percent_change = {row['mean_percent']:.2f}",
            f"  max_percent_change = {row['max_percent']:.2f}",
            f"  most_negative_location_lon = {row['most_negative_lon']:.4f}",
            f"  most_negative_location_lat = {row['most_negative_lat']:.4f}",
            "",
        ]
    )
stats_txt.write_text("\n".join(lines).rstrip() + "\n", encoding="utf-8")

print(f"Wrote {stats_csv}")
print(f"Wrote {stats_txt}")
stats



## 10. Plot Seasonal Percentage Change

The colour scale uses 5 percent intervals from -15 percent to 60 percent. The colour break around zero is kept light/white so decreases and increases are visually separated.


In [ ]:

def percentage_change_cmap(levels: np.ndarray) -> ListedColormap:
    interval_colors = []
    negative_palette = ["#6f6731", "#a09a64", "#d7cfaa"]
    positive_palette = [
        "#ffffff",
        "#dcecf0",
        "#c4d9e3",
        "#a9c4d5",
        "#88abc6",
        "#6f94b8",
        "#587ba3",
        "#4b6488",
        "#4d5272",
        "#504463",
        "#49385c",
        "#412c55",
    ]

    negative_index = 0
    positive_index = 0
    for low, high in zip(levels[:-1], levels[1:]):
        if high <= 0:
            color = negative_palette[min(negative_index, len(negative_palette) - 1)]
            negative_index += 1
        else:
            color = positive_palette[min(positive_index, len(positive_palette) - 1)]
            positive_index += 1
        interval_colors.append(color)
    return ListedColormap(interval_colors, name="rx1day_percentage_change")


def format_lon_ticks(values: np.ndarray) -> list[str]:
    return [f"{int(value)}?E" for value in values]


def format_lat_ticks(values: np.ndarray) -> list[str]:
    labels = []
    for value in values:
        if value == 0:
            labels.append("0?")
        elif value > 0:
            labels.append(f"{int(value)}?N")
        else:
            labels.append(f"{abs(int(value))}?S")
    return labels


def plot_region_boundary(ax, geometry) -> None:
    boundary = gpd.GeoSeries([geometry], crs="EPSG:4326").boundary
    boundary.plot(ax=ax, color="black", linewidth=0.7)


def plot_seasonal_change_maps(da: xr.DataArray, region_key: str) -> Path:
    levels = np.arange(-15, 65, 5)
    cmap = percentage_change_cmap(levels)
    norm = BoundaryNorm(levels, cmap.N)

    geometry = region_geometry(region_key)
    lon_min, lat_min, lon_max, lat_max = geometry.bounds
    extent = [lon_min - 0.6, lon_max + 0.6, lat_min - 0.4, lat_max + 0.4]

    fig, axes = plt.subplots(nrows=4, ncols=1, figsize=(9.0, 13.5), constrained_layout=False)
    fig.subplots_adjust(left=0.07, right=0.98, top=0.98, bottom=0.10, hspace=0.13)

    contours = []
    for ax, season in zip(axes, SEASONS):
        season_da = da.sel(season=season)
        contour = ax.contourf(
            season_da["lon"].values,
            season_da["lat"].values,
            season_da.values,
            levels=levels,
            cmap=cmap,
            norm=norm,
            extend="both",
        )
        contours.append(contour)
        plot_region_boundary(ax, geometry)
        ax.set_xlim(extent[0], extent[1])
        ax.set_ylim(extent[2], extent[3])
        ax.set_aspect("equal", adjustable="box")
        ax.text(
            0.5,
            0.96,
            season,
            transform=ax.transAxes,
            ha="center",
            va="top",
            fontsize=20,
            weight="bold",
        )
        xticks = np.arange(100, 121, 2)
        yticks = np.arange(0, 9, 2)
        ax.set_xticks(xticks)
        ax.set_yticks(yticks)
        ax.set_xticklabels(format_lon_ticks(xticks), fontsize=10)
        ax.set_yticklabels(format_lat_ticks(yticks), fontsize=10)
        ax.set_xlabel("")
        ax.set_ylabel("")

    cbar_ax = fig.add_axes([0.12, 0.04, 0.76, 0.020])
    cbar = fig.colorbar(
        contours[-1],
        cax=cbar_ax,
        orientation="horizontal",
        ticks=levels,
        spacing="proportional",
    )
    cbar.set_label("Percentage Change (%)", fontsize=11, weight="bold", labelpad=7)

    output_path = OUTPUT_DIR / f"rx1day_{region_key}_seasonal_ssp585_percent_change.png"
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    return output_path


plot_path = plot_seasonal_change_maps(seasonal_change, REGION_KEY)
plot_path



## 11. What To Try Next

- Change `REGION_KEY` to `"penisular"`, `"sabah"`, or `"sarawak"`.
- Start with one model, then switch `MODELS_TO_RUN` to multiple models or `"all"`.
- Set `USE_CACHE = False` if you want to force the notebook to recompute from raw NetCDF files.
- Edit the plot colour levels in Section 10 if the percentage-change range needs a different visual emphasis.
